# Clase 3 — Práctica: supremacía y simulación clásica

En esta práctica retomas el ejemplo de la cadena de Ising con campo transverso de la clase teórica y completas la comparación entre distintos números de repeticiones de Trotter.

## Enunciado

Ya tienes calculadas las curvas de magnetización con un paso de Trotter y con diez pasos. Tu tarea:

1. Genera los circuitos con tres repeticiones de Trotter (`trotter_lie(hamiltoniano, float(t), 3)`), a medio camino entre la versión de un paso y la de diez
2. Calcula la magnetización de esos circuitos con `capa_expectativa`
3. Dibuja las tres curvas juntas (uno, tres y diez pasos) en la misma gráfica
4. Anota, en una frase dentro de un comentario, qué relación observas entre el número de pasos de Trotter y el parecido con la curva de referencia (diez pasos)

### Pistas

- Usa la misma función `trotter_lie` que ya calculó las curvas de uno y diez pasos, solo cambia el número de repeticiones
- La capa `capa_expectativa` ya sabe evaluar una lista de circuitos junto con el operador `magnetizacion`
- Compara visualmente cuál de las dos curvas, la de uno o la de diez pasos, queda más cerca de la de tres pasos

In [ ]:
import cirq
import tensorflow_quantum as tfq
import numpy as np
import matplotlib.pyplot as plt

def cadena_ising_campo_transverso(n, campos):
    """Hamiltoniano de una cadena de n qubits con interacción Z-Z y campo X."""
    qubits = [cirq.GridQubit(0, i) for i in range(n)]
    hamiltoniano = cirq.PauliSum()
    for i in range(n - 1):
        hamiltoniano -= cirq.Z(qubits[i]) * cirq.Z(qubits[i + 1])
    for i in range(n):
        hamiltoniano -= campos[i] * cirq.X(qubits[i])
    return hamiltoniano

def paso_trotter(hamiltoniano, t):
    """Aproxima exp(-i * H * t) aplicando cada término del Hamiltoniano por separado."""
    return cirq.Circuit([tfq.util.exponential([t * termino]) for termino in hamiltoniano])

def trotter_lie(hamiltoniano, t, repeticiones):
    """Cuantas más repeticiones, más fino el paso y más preciso el resultado."""
    circuito = cirq.Circuit()
    for _ in range(repeticiones):
        circuito += paso_trotter(hamiltoniano, t / repeticiones)
    return circuito

n = 4
campos = [1.0] * n
hamiltoniano = cadena_ising_campo_transverso(n, campos)
qubits = [cirq.GridQubit(0, i) for i in range(n)]
magnetizacion = sum(cirq.Z(q) for q in qubits)

tiempos = np.linspace(0.01, 2, 20)

# Ya calculadas: las curvas de referencia de 1 y 10 repeticiones de Trotter
circuitos_1_paso = [trotter_lie(hamiltoniano, float(t), 1) for t in tiempos]
circuitos_10_pasos = [trotter_lie(hamiltoniano, float(t), 10) for t in tiempos]

capa_expectativa = tfq.layers.Expectation()
exp_1_paso = capa_expectativa(circuitos_1_paso, symbol_names=[], symbol_values=[[] for _ in tiempos], operators=magnetizacion).numpy()
exp_10_pasos = capa_expectativa(circuitos_10_pasos, symbol_names=[], symbol_values=[[] for _ in tiempos], operators=magnetizacion).numpy()

# TODO: genera los circuitos con 3 repeticiones de Trotter, igual que hiciste con 1 y 10
circuitos_3_pasos = None

# TODO: calcula la magnetización de esos circuitos con capa_expectativa
exp_3_pasos = None

# TODO: dibuja las tres curvas juntas (1, 3 y 10 pasos) con plt.plot, plt.xlabel, plt.ylabel y plt.legend

# TODO: en un comentario, anota qué relación observas entre el número de pasos de Trotter
# y el parecido de cada curva con la curva de referencia (10 pasos)

<details>
<summary>💡 Ver solución</summary>

```python
# Generas los circuitos con 3 repeticiones de Trotter, a medio camino entre 1 y 10
circuitos_3_pasos = [trotter_lie(hamiltoniano, float(t), 3) for t in tiempos]

# Calculas la magnetización de esos circuitos con la misma capa de TensorFlow Quantum
exp_3_pasos = capa_expectativa(circuitos_3_pasos, symbol_names=[], symbol_values=[[] for _ in tiempos], operators=magnetizacion).numpy()

# Dibujas las tres curvas juntas para compararlas
plt.plot(tiempos, exp_1_paso, label="1 paso de Trotter")
plt.plot(tiempos, exp_3_pasos, label="3 pasos de Trotter")
plt.plot(tiempos, exp_10_pasos, label="10 pasos de Trotter")
plt.xlabel("Tiempo")
plt.ylabel("Magnetización")
plt.legend()
plt.show()

# Relación observada: cuantas más repeticiones de Trotter usas, más se acerca la curva
# a la de referencia (10 pasos); la curva de 3 pasos queda claramente más cerca de la
# de 10 pasos que la de 1 paso, porque un paso más fino aproxima mejor la evolución exacta
```

</details>